In [27]:
import brightway2 as bw
import pandas as pd
import openpyxl 
import os
import glob

In [28]:
bw.projects.set_current('Prospective LCA v1')

In [29]:
databaseNames = bw.databases
myDatabaseNames = []
ecoInventDatabaseNames = []
for databaseName in databaseNames:
    if 'Abhi' in databaseName:
        myDatabaseNames.append(databaseName)
    elif 'Abhi' not in databaseName and 'biosphere3' not in databaseName and ('image' in databaseName or 'remind' in databaseName):
        ecoInventDatabaseNames.append(databaseName)
myDatabaseNames.sort()
ecoInventDatabaseNames.sort()

In [30]:
GWPMethod = [method for method in bw.methods if 'IPCC 2013' in str(method) and  'climate change' 
                in str(method) and 'GWP 100a' in str(method) and not 'no LT' in str(method)]

In [31]:
def breakdown_calculations(db, activity):
    activitiesList = [{activity : 1}]
    for exchange in activity.technosphere():
        activitiesList.append({bw.Database(exchange.input.key[0]).get(exchange.input.key[1]) : exchange.amount})
    calculationSetup = {'inv' : activitiesList, 'ia' : GWPMethod}
    bw.calculation_setups['breakdown'] = calculationSetup
    myLCA = bw.MultiLCA('breakdown')
    results = pd.DataFrame(myLCA.results.transpose(), columns = [str(list(i.keys())[0]).split('\'')[1] for i in activitiesList], index = pd.MultiIndex.from_tuples(GWPMethod))
    results = results.sort_index().drop(index = [i for i in results.index if i[0] == 'rest'])
    directEmissions = pd.DataFrame([0 if abs(results.iloc[r,0] - results.iloc[r,1:].sum()) / abs(results.iloc[r,0]) < 1e-5 else results.iloc[r, 0] - results.iloc[r, 1:].sum() for r in range(len(results.index))],
                                      columns = ['direct emissions'],
                                      index = results.index)
    results = pd.concat([results, directEmissions], axis = 1)
    results['database'] = db.name
    return results

In [32]:
result_dataframes = {
    'hydrogenBAU': pd.DataFrame(),
    'hydrogenWind': pd.DataFrame(),
    'hydrogenSolar': pd.DataFrame(),
    'ammoniaBAU': pd.DataFrame(),
    'ammoniaWind': pd.DataFrame(),
    'ammoniaSolar': pd.DataFrame(),
    'methanolBAU': pd.DataFrame(),
    'methanolWind': pd.DataFrame(),
    'methanolSolar': pd.DataFrame(),
    'ethyleneBAU': pd.DataFrame(),
    'ethyleneWind': pd.DataFrame(),
    'ethyleneSolar': pd.DataFrame()
}

In [33]:
for i in range(0, len(myDatabaseNames)):
    
    myDB = bw.Database(myDatabaseNames[i])
    myActivities = list(myDB)
    ecoInventDB = bw.Database(ecoInventDatabaseNames[i])
    ecoInventActivities = list(ecoInventDB)

    hydrogenBAUActivity = next(filter(
        lambda x: 'hydrogen, from steam methane reforming' in x['name']
                    and 'hydrogen, Abhi' in x['reference product']
                    and 'CCS' not in x['name']
                    and 'biomethane' not in x['name']
                    and 'GLO' in x['location'],
        myActivities
    ))
    hydrogenWindActivity = next(filter(
        lambda x: 'hydrogen, PEM electrolysis, wind power, >3MW, onshore' in x['name']
                    and 'hydrogen, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))
    hydrogenSolarActivity = next(filter(
        lambda x: 'hydrogen, PEM electrolysis, solar power' in x['name']
                    and 'hydrogen, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))
    ammoniaBAUActivity = [activity for activity in ecoInventDB if 'ammonia production, steam reforming, liquid' in activity['name']
                                                                and 'RoW' in activity['location']
                                                                and 'ammonia, anhydrous, liquid' in activity['reference product']][0]
    ammoniaWindActivity = next(filter(
        lambda x: 'ammonia; hydrogen, PEM electrolysis, wind power, >3MW, onshore' in x['name']
                    and 'ammonia, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))
    ammoniaSolarActivity = next(filter(
        lambda x: 'ammonia; hydrogen, PEM electrolysis, solar power' in x['name']
                    and 'ammonia, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))
    methanolBAUActivity = next(filter(
        lambda x: 'methanol production' in x['name']
                    and 'GLO' in x['location'],
        ecoInventActivities
    ))
    methanolWindActivity = next(filter(
        lambda x: 'methanol; carbon dioxide, DACCS, carbon engineering, configuration A; hydrogen, PEM electrolysis, wind power, >3MW, onshore' in x['name']
                    and 'methanol, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))
    methanolSolarActivity = next(filter(
        lambda x: 'methanol; carbon dioxide, DACCS, carbon engineering, configuration A; hydrogen, PEM electrolysis, solar power' in x['name']
                    and 'methanol, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))
    ethyleneBAUActivity = next(filter(
        lambda x: 'ethylene, fMTO; methanol, BAU' in x['name']
                    and 'ethylene, Abhi' in x['reference product']
                    and 'CCS' not in x['name']
                    and 'biomethane' not in x['name']
                    and 'GLO' in x['location'],
        myActivities
    ))
    ethyleneWindActivity = next(filter(
        lambda x: 'ethylene, gMTO; methanol; carbon dioxide, DACCS, carbon engineering, configuration A; hydrogen, PEM electrolysis, wind power, >3MW, onshore' in x['name']
                    and 'ethylene, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))
    ethyleneSolarActivity = next(filter(
        lambda x: 'ethylene, gMTO; methanol; carbon dioxide, DACCS, carbon engineering, configuration A; hydrogen, PEM electrolysis, solar power' in x['name']
                    and 'ethylene, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        myActivities
    ))

    result_dataframes['hydrogenBAU'] = pd.concat(
        [result_dataframes['hydrogenBAU'], breakdown_calculations(myDB, hydrogenBAUActivity)],
        ignore_index = True
    )
    result_dataframes['hydrogenWind'] = pd.concat(
        [result_dataframes['hydrogenWind'], breakdown_calculations(myDB, hydrogenWindActivity)],
        ignore_index = True
    )
    result_dataframes['hydrogenSolar'] = pd.concat(
        [result_dataframes['hydrogenSolar'], breakdown_calculations(myDB, hydrogenSolarActivity)],
        ignore_index = True
    )
    result_dataframes['ammoniaBAU'] = pd.concat(
        [result_dataframes['ammoniaBAU'], breakdown_calculations(ecoInventDB, ammoniaBAUActivity)],
        ignore_index = True
    )
    result_dataframes['ammoniaWind'] = pd.concat(
        [result_dataframes['ammoniaWind'], breakdown_calculations(myDB, ammoniaWindActivity)],
        ignore_index = True
    )
    result_dataframes['ammoniaSolar'] = pd.concat(
        [result_dataframes['ammoniaSolar'], breakdown_calculations(myDB, ammoniaSolarActivity)],
        ignore_index = True
    )
    result_dataframes['methanolBAU'] = pd.concat(
        [result_dataframes['methanolBAU'], breakdown_calculations(ecoInventDB, methanolBAUActivity)],
        ignore_index = True
    )
    result_dataframes['methanolWind'] = pd.concat(
        [result_dataframes['methanolWind'], breakdown_calculations(myDB, methanolWindActivity)],
        ignore_index = True
    )
    result_dataframes['methanolSolar'] = pd.concat(
        [result_dataframes['methanolSolar'], breakdown_calculations(myDB, methanolSolarActivity)],
        ignore_index = True
    )
    result_dataframes['ethyleneBAU'] = pd.concat(
        [result_dataframes['ethyleneBAU'], breakdown_calculations(myDB, ethyleneBAUActivity)],
        ignore_index = True
    )
    result_dataframes['ethyleneWind'] = pd.concat(
        [result_dataframes['ethyleneWind'], breakdown_calculations(myDB, ethyleneWindActivity)],
        ignore_index = True
    )
    result_dataframes['ethyleneSolar'] = pd.concat(
        [result_dataframes['ethyleneSolar'], breakdown_calculations(myDB, ethyleneSolarActivity)],
        ignore_index = True
        )

In [34]:
for result_dataframe in result_dataframes:
   df = result_dataframes[result_dataframe]
   df['name'] = df.columns[0]
   df.set_index('name')
   last_column = df.pop(df.columns[-1])
   df.insert(0, last_column.name, last_column)
   database_column = df.pop(df.columns[-1])
   df.insert(1, database_column.name, database_column)

In [39]:
result_dataframes['ammoniaBAU']['database'] = result_dataframes['ammoniaBAU']['database'].str.replace('ecoinvent 3.8 cutoff', 'lci-Abhi', regex = True)
result_dataframes['methanolBAU']['database'] = result_dataframes['methanolBAU']['database'].str.replace('ecoinvent 3.8 cutoff', 'lci-Abhi', regex = True)

In [40]:
for result_dataframe in result_dataframes:
   df = result_dataframes[result_dataframe]
   df['database'] = df['database'].str.replace('Base', 'RCP6')
   df['database'] = df['database'].str.replace('PkBudg500', 'RCP19')
   df['database'] = df['database'].str.replace('PkBudg1150', 'RCP26')

In [41]:
hydrogenResultsFileName = 'hydrogen_GWP_results_breakdown.xlsx'
ammoniaResultsFileName = 'ammonia_GWP_results_breakdown.xlsx'
methanolResultsFileName = 'methanol_GWP_results_breakdown.xlsx'
ethyleneResultsFileName = 'ethylene_GWP_results_breakdown.xlsx'

In [42]:
def check_or_create_excel_file(filePath):
    if not os.path.exists(filePath):
        wb = openpyxl.Workbook()
        wb.save(filePath)
        print(f"Excel file created at {filePath}")

def write_dataframes_to_excel(dictionary, file_name):
    check_or_create_excel_file(file_name)
    with pd.ExcelWriter(file_name, engine = 'openpyxl', mode = 'a') as writer:        
        for sheet_name, dataframe in dictionary.items():
            if sheet_name in writer.book.sheetnames:
                writer.book.remove(writer.book[sheet_name])
            dataframe.to_excel(writer, sheet_name = sheet_name)
    wb = openpyxl.load_workbook(file_name)
    if wb.sheetnames[0] == 'Sheet':  # check if the workbook has any sheets
        firstSheet = wb.sheetnames[0]  # get the name of the first sheet
        wb.remove(wb[firstSheet])  # remove the first sheet
        wb.save(file_name)

In [43]:
hydrogen_dataframes = {}
ammonia_dataframes = {}
methanol_dataframes = {}
ethylene_dataframes = {}
for key in list(result_dataframes.keys()):
    if 'hydrogen' in key:
        hydrogen_dataframes[key] = result_dataframes[key]
    elif 'ammonia' in key:
        ammonia_dataframes[key] = result_dataframes[key]
    elif 'methanol' in key:
        methanol_dataframes[key] = result_dataframes[key]
    elif 'ethylene' in key:
        ethylene_dataframes[key] = result_dataframes[key]

write_dataframes_to_excel(hydrogen_dataframes, os.path.join('..', 'Results', 'Hydrogen', hydrogenResultsFileName))
write_dataframes_to_excel(ammonia_dataframes, os.path.join('..', 'Results', 'Ammonia', ammoniaResultsFileName))
write_dataframes_to_excel(methanol_dataframes, os.path.join('..', 'Results', 'Methanol', methanolResultsFileName))
write_dataframes_to_excel(ethylene_dataframes, os.path.join('..', 'Results', 'Ethylene', ethyleneResultsFileName))

Excel file created at ..\Results\Hydrogen\hydrogen_GWP_results_breakdown.xlsx
Excel file created at ..\Results\Ammonia\ammonia_GWP_results_breakdown.xlsx
Excel file created at ..\Results\Methanol\methanol_GWP_results_breakdown.xlsx
Excel file created at ..\Results\Ethylene\ethylene_GWP_results_breakdown.xlsx
